# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, with explicit references to each entity's `@id`, following best practices for FAIR data. We focus on using the Croissant schema hosted at [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata attributes
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\nVersion: {metadata.version}")

## 2. Data Overview
Review available record sets, their `@id` values, and the fields/columns for each.

We explicitly refer to all record sets and fields/columns by their `@id`, as per best practices for working with Croissant datasets.

In [ ]:
# List all RecordSets with their @id and fields
print('Record Sets with their @id and fields:')
record_sets = []
for rs in metadata.record_sets:
    print(f"- RecordSet @id: {rs.id}, name: {getattr(rs, 'name', '<no name>')}")
    record_sets.append(rs.id)
    for field in rs.fields:
        print(f"    - Field @id: {field.id}, name: {getattr(field,'name','<no name>')}")
        # Fields can have columns
        if hasattr(field, 'columns') and field.columns is not None:
            for column in field.columns:
                print(f"      - Column @id: {column.id}, name: {getattr(column,'name','<no name>')} ({getattr(column, 'data_type', '<no type>')})")

## 3. Data Extraction
Select a specific record set (using its `@id`) and load all records into a pandas DataFrame. You can adjust the selection of `record_set_id` and field/column as desired from the previous cell's output.

In [ ]:
# For this dataset, pick the first record set by @id. Adjust to your needs if more are present.
record_set_id = record_sets[0]
print(f"Loading records from RecordSet @id: {record_set_id}")

# Load records as a list of dicts and convert to DataFrame
records = list(dataset.records(record_set=record_set_id))
df = pd.DataFrame(records)
print(f"Available columns (by field @id):\n{df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. You can adjust the `numeric_field_id` and `group_field_id` based on the field overview above. All fields are referenced by their `@id`.

In [ ]:
# Example field @id selection (adjust as needed based on prior listing)
# You might see fields like 'cr:protein-abundance', 'cr:molecular-weight', etc.

# Choose a numeric field for analysis, e.g., Molecular Weight (replace with actual @id if different)
numeric_field_id = None
for col in df.columns:
    if 'weight' in col.lower() or 'mw' in col.lower():
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Fallback: just take the first numeric field
    for col in df.select_dtypes(include='number').columns:
        numeric_field_id = col
        break
if numeric_field_id is None:
    raise Exception('No obvious numeric field for analysis. Please select one from df.columns.')
print(f"Analysing numeric field (by @id): {numeric_field_id}")

# Filtering: E.g., select records with numeric_field > 10000
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered {len(filtered_df)} records where {numeric_field_id} > mean ({threshold:.2f})")

    # Normalization
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"First 5 normalized values:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Grouping by another field (e.g., 'cr:description' or any descriptive @id)
    possible_group_fields = [col for col in df.columns if 'description' in col.lower() or 'accession' in col.lower() or 'group' in col.lower()]
    if possible_group_fields:
        group_field_id = possible_group_fields[0]
        print(f"Grouping by field (by @id): {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame().reset_index()
        print(grouped_df.head())
    else:
        print("No obvious field to group by.")
else:
    print(f"Field {numeric_field_id} is not numeric. Please choose another field for filtering and analysis.")

## 5. Visualization
Visualize data for the selected fields using matplotlib or seaborn. Make sure to use the field `@id` as the column label.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of the selected numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If we have a group field, plot mean values by group
if 'group_field_id' in locals():
    plt.figure(figsize=(10,5))
    plot_df = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(20)
    sns.barplot(x=plot_df.values, y=plot_df.index, orient='h')
    plt.title(f"Mean {numeric_field_id} by {group_field_id} (top 20)")
    plt.xlabel(f"Mean {numeric_field_id}")
    plt.ylabel(group_field_id)
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to:
- Load and inspect Croissant-based FAIR datasets using the `mlcroissant` library,
- Reference all entities (record sets, fields, columns) explicitly by their `@id`,
- Extract structured records into pandas DataFrames for analysis,
- Apply common EDA transformations (filtering, normalization, grouping) to analyze protein data,
- Visualize key statistical distributions for further insights.

**Next steps:** You can further explore relationships between protein abundance, molecular weights, or other fields by adjusting the `@id` references in EDA and plotting. For detailed schema, always consult the Croissant JSON-LD or the printed listing of `@id` values above.